In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader 
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torchvision import transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)

In [3]:
train_loader=DataLoader(trainset,batch_size=64,shuffle=True)
test_loader=DataLoader(testset,batch_size=64)

In [22]:
#Building the model
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),#input channels,no_of_total_filters,kernal-size,padding
            nn.ReLU(),
            nn.MaxPool2d(2,2),#kernal size,stride

            nn.Conv2d(32,64,kernel_size=3,padding=1),#input channels,no_of_total_filters,kernal-size,padding
            nn.ReLU(),
            nn.MaxPool2d(2,2),#kernal size,stride

            nn.Conv2d(64,128,kernel_size=3,padding=1),#input channels,no_of_total_filters,kernal-size,padding
            nn.ReLU(),
            nn.MaxPool2d(2,2),#kernal size,stride
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256), #after pooling it decreases the width and height by half so we get 4*4*128
            nn.ReLU(),

            nn.Linear(256,10), #we are using multiclass_classification so we have 10 outputs on which softmax will be applied by default
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1) #falttening code
        x=self.fc_layers(x)
        return x

In [23]:
model=CNN()
criterian=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [24]:
#Training the model
epoches=10

for epoch in range(epoches):
    training_loss=0.0

    for images,labels in train_loader:
        optimizer.zero_grad()

        output=model.forward(images)
        loss=criterian(output,labels)
        loss.backward()
        optimizer.step()

        training_loss+=loss.item()

    print("epoch:",epoch+1,"/",epoches,"--> loss:",training_loss/len(train_loader))

epoch: 1 / 10 --> loss: 1.3520066382177651
epoch: 2 / 10 --> loss: 0.9238059039768356
epoch: 3 / 10 --> loss: 0.7406816533992967
epoch: 4 / 10 --> loss: 0.6163208955312933
epoch: 5 / 10 --> loss: 0.5039363806052586
epoch: 6 / 10 --> loss: 0.4101550111837704
epoch: 7 / 10 --> loss: 0.3196735206200644
epoch: 8 / 10 --> loss: 0.2425807348697844
epoch: 9 / 10 --> loss: 0.18894191714160888
epoch: 10 / 10 --> loss: 0.14525877625045494


In [27]:
#Evaluation of cnn
correct=0
total=0

model.eval()

with torch.no_grad():
    for images,labels in test_loader:
        output=model(images)
        _,prediction=torch.max(output,1)

        correct+=(prediction==labels).sum().item()
        total+=labels.size(0)

    print("accuracy:",correct/total)
           

accuracy: 0.7281
